In [2]:
import os
from pathlib import Path
from tqdm import tqdm
import pandas as pd

from shapely.geometry import box

import sys
sys.path.append('..')  # Adjust the path as per your directory structure

from scripts.constants import *
from scripts.utils import *
from scripts.raster_operations import *
from scripts.vector_operations import *

In [13]:
# IN paths
bua_path = BUA_IN_DIR / "OS_Open_Built_Up_Areas.gpkg"
pch_vrt_path = PCH_IN_DIR / "ps_PSScene4Band_2019.vrt"
pch_paths = list(sorted(PCH_IN_DIR.glob('*.tif')))
gbg_5km_path = GBG_IN_DIR / "5km_grid_region.shp"
bhf_paths = list(sorted(BHF_IN_DIR.rglob('*9.gdbtable')))
ogs_path = OGS_IN_DIR / "GB_GreenspaceSite.shp"
imd_data_path = IMD_IN_DIR / "uk_imd2019.csv"
imd_eng_wal_bound_path = IMD_IN_DIR / "England_Wales_LSOA/5878ebcf-f091-4bde-8864-aa547dd45afd2020330-1-8g4usn.8evuq.shp"
imd_sco_bound_path = IMD_IN_DIR / "Scotland_datazone/SG_DataZone_Bdry_2011.shp"

# OUT paths
bua_dissolved_path = BUA_OUT_DIR / "bua_dissolved.geojson"
pch_binary_dir = PCH_OUT_DIR / "pch_binary"
pch_binary_dir.mkdir(parents=True, exist_ok=True)
pch_binary_vrt_path = pch_binary_dir / "ps_PSScene4Band_2019_binary.vrt"
pch_masked_dir = PCH_OUT_DIR / "pch_binary"
pch_masked_dir.mkdir(parents=True, exist_ok=True)
pch_masked_vrt_path = pch_masked_dir / "ps_PSScene4Band_2019_masked.vrt"
pch_masked_tile_dir = PCH_OUT_DIR / "pch_bua_masked_gbg_tile"
pch_masked_r_tile_dir = pch_masked_tile_dir / "raster_tile"
pch_masked_r_tile_dir.mkdir(parents=True, exist_ok=True)
pch_masked_v_tile_dir = pch_masked_tile_dir / "vector_tile"
pch_masked_v_tile_dir.mkdir(parents=True, exist_ok=True)
bhf_dissolved_tile_dir = BHF_OUT_DIR / "bhf_dissolved_tile"
bhf_dissolved_tile_dir.mkdir(parents=True, exist_ok=True)
bhf_distance_tile_dir = BHF_OUT_DIR / "bhf_distance_tile"
bhf_distance_tile_dir.mkdir(parents=True, exist_ok=True)
imd_gb_gdf_path = IMD_OUT_DIR / "imd_gb_merged.geojson"

# Data Processing

## PCH and BUA Processing

### Create VRT from PCH

In [43]:
create_vrt(pch_paths, pch_vrt_path)

### Dissolve BUA

In [11]:
bua_gpd = gpd.read_file(bua_path)
bua_crs = bua_gpd.crs
bua_dissolved_gpd = bua_gpd.dissolve().reset_index(drop=True).drop(columns=['gsscode', 'name1_text', 'name1_language', 'name2_text',
       'name2_language', 'areahectares', 'geometry_area_m'])

Handle everything with pch's crs, which is EPSG:4326

In [ ]:
with rasterio.open(pch_vrt_path) as pch:
    global pch_crs
    pch_crs = pch.crs
    bua_dissolved_gpd = bua_dissolved_gpd.to_crs(pch_crs)
    bua_dissolved_gpd.to_file(bua_dissolved_path)

### Mask PCH with BUA dissolved

In [9]:
mask_path = bua_dissolved_path
min_val, max_val = 3, 70

for i, in_path in tqdm(enumerate(pch_paths)):
    
    out_name = in_path.stem + "_masked.tif"
    out_path = pch_masked_dir / out_name
    
    clip_and_mask_raster(in_path, bua_dissolved_gpd, out_path, min_val, max_val)

82it [22:33, 16.50s/it]


#### Create VRT of PCH-masked

In [14]:
pch_masked_paths = list(sorted(pch_masked_dir.glob('*.tif')))
create_vrt(pch_masked_paths, pch_masked_vrt_path)

### Binary Mask

In [11]:
import rasterio
import numpy as np
from rasterio.mask import mask
from rasterio.features import geometry_mask

def binarise_raster(raster_path, output_path, min_val, max_val):
    """
    Clips a raster with a vector mask, binarizes values within the specified range,
    and saves the output to a new binary TIFF file.

    Parameters:
        raster_path (str): Path to the raster file.
        vector_path (str): Path to the vector mask file.
        output_path (str): Where to save the clipped and masked raster.
        min_val (float): Minimum value to keep and binarize to 1.
        max_val (float): Maximum value to keep and binarize to 1.
    """
    with rasterio.open(raster_path) as src:
        data = src.read(1)
        # Binarize the image: 1 for values within range, 0 otherwise
        binary_data = np.where(
            (data >= min_val) & (data <= max_val),
            1,  # Values within the range are set to 1
            0   # Values outside the range are set to 0
        )
        
        # Prepare metadata for the output file
        out_meta = src.meta.copy()
        out_meta.update({
            "driver": "GTiff",
            "dtype": "uint8",
            "count": 1,
            "nodata": None  # No nodata value for a binary raster
        })
    
    # Write the binarized raster to a new file
    with rasterio.open(output_path, "w", **out_meta) as dest:
        dest.write(binary_data, 1)  # Write the binarized data as the first (and only) band


# Example usage
# Note: Ensure you have the `vector_mask` as a GeoDataFrame or similar object that has a 'geometry' field


In [12]:
mask_path = bua_dissolved_path
min_val, max_val = 3, 70

for i, in_path in tqdm(enumerate(pch_paths)):
    
    out_name = in_path.stem + "_binary.tif"
    out_path = pch_binary_dir / out_name
    
    binarise_raster(in_path, out_path, 3, 70)

82it [05:29,  4.01s/it]


In [14]:
pch_binary_paths = list(sorted(pch_binary_dir.glob('*.tif')))
create_vrt(pch_binary_paths, pch_binary_vrt_path)

/Users/ancazugo/Documents/PhD_Thesis/Tree_detection/.venv/lib/python3.12/site-packages/osgeo/gdal.py:312: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


### Tile PCH-masked using GBG tiles (5 km)

In [7]:
gbg_5km_gdf = gpd.read_file(gbg_5km_path)
gbg_crs = gbg_5km_gdf.crs
gbg_gdf = gbg_5km_gdf.copy().to_crs(pch_crs).sort_values(by=['TILE_NAME'])

In [128]:
def vectorise_raster(raster_data, transform, field_name, crs, dissolve=True):
    # Generate vector shapes from raster
    mask = raster_data != raster_data.min()  # Mask to exclude the background/nodata values
    vector_shapes = shapes(raster_data, mask=mask, transform=transform)

    # Convert the shapes and values to a GeoDataFrame
    records = []
    for geom, value in vector_shapes:
        records.append({
            'geometry': shape(geom),
            field_name: value
        })

    # Convert the shapes and values to a GeoDataFrame
    if records:  # Check if there are any records to avoid creating an empty GeoDataFrame
        gdf = gpd.GeoDataFrame(records, crs=crs)
        gdf.set_geometry('geometry', inplace=True)  # Ensure 'geometry' is the geometry column
        gdf = gdf  # Assign CRS after confirming geometry presence

        # Optional dissolve step
        if dissolve:
            gdf = gdf.dissolve(by=field_name, as_index=False)

        return gdf
    else:
        return None  # Return None if no vector shapes were created

In [147]:
gbg_gdf[gbg_gdf['TILE_NAME'] == 'ST72NW'] # Desde ST72NW no tienen tif file
print(len(gbg_gdf))

10647


In [ ]:
# Open the raster file
error_tiles =[]
empty_tiles = []
empty_vectors = []
with rasterio.open(pch_masked_vrt_path) as src:
    gbg_gdf = gbg_gdf.to_crs(src.crs)
    for index, row in tqdm(gbg_gdf.iloc[7240:].iterrows()):
        # if index > 10:
        #     break
        tile_name = row.TILE_NAME
        output_name = f"pch_{tile_name}"
        
        # The geometry to mask the raster with

        # geom = getFeatures(gpd.GeoDataFrame([row], crs=src.crs))
        # geom = [row['geometry']] # Opcion 1 --> funciona
        temp_gdp = gpd.GeoDataFrame([row], crs=src.crs)
        geom = [feature["geometry"] for _, feature in temp_gdp.iterrows()]
        
        try:
            # print('masking...')
            out_image, out_transform = mask(src, geom, crop=True)
            if out_image.size == 0 or out_image.mean() == src.nodata:
                # print(index, "Empty raster tile", tile_name)
                empty_tiles.append(tile_name)
                continue
            # print('vectorising...')
            vector_gdf = vectorise_raster(out_image[0], out_transform, 'height', src.crs, dissolve=True)
        
            # Save vectorized output
            if vector_gdf is None:
                # print(index, "Empty vector tile", tile_name)
                empty_vectors.append(tile_name)
                continue
            print(tile_name)
            
            tile_r_code_dir = pch_masked_r_tile_dir / tile_name[:2].lower()
            tile_r_code_dir.mkdir(parents=True, exist_ok=True)
            output_r_path = tile_r_code_dir / f"{output_name}.tif"
            
            tile_v_code_dir = pch_masked_v_tile_dir / tile_name[:2].lower()
            tile_v_code_dir.mkdir(parents=True, exist_ok=True)
            output_v_path = tile_v_code_dir / f"{output_name}.geojson"
            vector_gdf.to_file(output_v_path, driver='GeoJSON')

            out_meta = src.meta.copy()

            # Update the metadata to reflect the number of layers,
            # and the new transformed affine and the new height and width
            out_meta.update({"driver": "GTiff",
                                "height": out_image.shape[1],
                                "width": out_image.shape[2],
                                "transform": out_transform})
            
            # with rasterio.open(output_r_path, "w", **out_meta) as dest:
            #     dest.write(out_image)
        
        except ValueError as e:
            print(index, e, tile_name)
            error_tiles.append(tile_name)

## BHF Processing

Dissolve BHF with itself keeping non-adjacent polygons as separate features

In [154]:
for bhf_path in tqdm(bhf_paths[0:1]):
    
    bhf_tile_out_dir = bhf_dissolved_tile_dir / bhf_path.parents[1].name
    bhf_tile_out_dir.mkdir(parents=True, exist_ok=True)
    bhf_tile_out_path = bhf_tile_out_dir / f"bhf_dissolved_{bhf_path.parent.name[:6]}.geojson"
    
    bhf_gpd = gpd.read_file(bhf_path)

    bhf_dissolved_gpd = dissolve_adjacent(bhf_gpd).to_crs(pch_crs)

    bhf_dissolved_gpd.to_file(bhf_tile_out_path)

bhf_crs = bhf_gpd.crs

100%|██████████| 1/1 [00:00<00:00, 149.32it/s]


## OGS Processing

In [3]:
ogs_gdf = gpd.read_file(ogs_path)
ogs_crs = ogs_gdf.crs

In [16]:
ogs_gdf = ogs_gdf.copy()[ogs_gdf['function'] == 'Public Park Or Garden'].to_crs(4326)
for i, row in tqdm(gbg_gdf.iterrows()):
    # Assuming we're using the first feature in the clipping layer
    tile_name = row.TILE_NAME
    ogs_tile_out_dir = OGS_OUT_DIR / tile_name[:2].lower()
    ogs_tile_out_dir.mkdir(parents=True, exist_ok=True)
    ogs_tile_out_path = ogs_tile_out_dir / f"ogs_clipped_{tile_name}.geojson"
    
    bounding_box = row.geometry.bounds
    # Convert the bounding box to a polygon
    clip_extent_polygon = box(*bounding_box)
    # Clip the layer
    ogs_clipped_gdf = gpd.clip(ogs_gdf, clip_extent_polygon)

    if len(ogs_clipped_gdf) > 0:

        ogs_clipped_gdf.to_file(ogs_tile_out_path)

10647it [00:10, 1060.12it/s]


## IMD Processing

Merge the Scotland and England and Wales boundaries and join them to the IMD 2019 data with the LSOA/Data Zone codes

In [15]:
imd_df = pd.read_csv(imd_data_path)
imd_eng_wal_bound_gdf = gpd.read_file(imd_eng_wal_bound_path)
imd_eng_wal_bound_crs = imd_eng_wal_bound_gdf.crs
imd_sco_bound_gdf = gpd.read_file(imd_sco_bound_path)
imd_sco_bound_crs = imd_sco_bound_gdf.crs

In [31]:
imd_gb_gdf = pd.concat([imd_eng_wal_bound_gdf.copy().to_crs(4326), 
                        imd_sco_bound_gdf.copy().to_crs(4326)]).merge(imd_df, on='LSOA', how='inner')

imd_gb_gdf.to_file(imd_gb_gdf_path)

# 3-30-300 Estimations

Create a DataFrame that gathers all the file paths generated in the processing steps to determine which tiles have the three types of data:

- PCH masked vector tiles
- BHF dissolved tiles
- OGS clipped tiles

Note that GBG has two systems for naming tiles, which are translated using the `translate_tile_name()` function from `utils.py`

In [18]:
pch_tile_paths = list(sorted(pch_masked_v_tile_dir.rglob('*.geojson')))
bhf_tile_paths = list(sorted(bhf_dissolved_tile_dir.rglob('*.geojson')))
ogs_tile_paths = list(sorted(OGS_OUT_DIR.rglob('*.geojson')))

In [19]:
pch_tile_paths_df = pd.DataFrame([{'big_tile': p.stem[-6:-4].lower(), 'direction_code': p.stem[-6:], 'path': str(p)} for p in pch_tile_paths])
pch_tile_paths_df['number_code'] = pch_tile_paths_df.apply(lambda x: translate_tile_name(x.direction_code), axis=1)

bhf_tile_paths_df = pd.DataFrame([{'big_tile': p.stem[-6:-4], 'number_code': p.stem[-6:], 'path': str(p)} for p in bhf_tile_paths])
bhf_tile_paths_df['direction_code'] = bhf_tile_paths_df.apply(lambda x: translate_tile_name(x.number_code), axis=1)

ogs_tile_paths_df = pd.DataFrame([{'big_tile': p.parent.stem, 'direction_code': p.stem[-6:], 'path': str(p)} for p in ogs_tile_paths])
ogs_tile_paths_df['number_code'] = ogs_tile_paths_df.apply(lambda x: translate_tile_name(x.direction_code), axis=1)

In [20]:
columns = ['big_tile', 'number_code', 'direction_code', 'path_pch', 'path_bhf', 'path_ogs']
tile_paths_df = pch_tile_paths_df.merge(
    bhf_tile_paths_df.merge(ogs_tile_paths_df, how='outer', on=['big_tile', 'direction_code', 'number_code'], suffixes=['_bhf', '_ogs']),
    how='outer', on=['big_tile', 'direction_code', 'number_code']
    ).rename(columns={'path': 'path_pch'})[columns]
tile_paths_df

,big_tile,number_code,direction_code,path_pch,path_bhf,path_ogs
0,hp,hp4500,HP40SE,NaN,/Users/ancazugo/Documents/PhD_Thesis/Tree_dete...,NaN
1,hp,hp5505,HP50NE,NaN,/Users/ancazugo/Documents/PhD_Thesis/Tree_dete...,NaN
2,hp,hp5005,HP50NW,NaN,/Users/ancazugo/Documents/PhD_Thesis/Tree_dete...,NaN
3,hp,hp5500,HP50SE,NaN,/Users/ancazugo/Documents/PhD_Thesis/Tree_dete...,NaN
4,hp,hp5000,HP50SW,NaN,/Users/ancazugo/Documents/PhD_Thesis/Tree_dete...,NaN
...,...,...,...,...,...,...
8356,tv,tv4595,TV49NE,/Users/ancazugo/Documents/PhD_Thesis/Tree_dete...,/Users/ancazugo/Documents/PhD_Thesis/Tree_dete...,/Users/ancazugo/Documents/PhD_Thesis/Tree_dete...
8357,tv,tv4095,TV49NW,NaN,/Users/ancazugo/Documents/PhD_Thesis/Tree_dete...,NaN
8358,tv,tv5595,TV59NE,/Users/ancazugo/Documents/PhD_Thesis/Tree_dete...,/Users/ancazugo/Documents/PhD_Thesis/Tree_dete...,/Users/ancazugo/Documents/PhD_Thesis/Tree_dete...
8359,tv,tv5095,TV59NW,/Users/ancazugo/Documents/PhD_Thesis/Tree_dete...,/Users/ancazugo/Documents/PhD_Thesis/Tree_dete...,/Users/ancazugo/Documents/PhD_Thesis/Tree_dete...


In [21]:
tile_paths_filtered_df = tile_paths_df.copy()[~tile_paths_df['path_bhf'].isna()].reset_index(drop=True)
tile_paths_filtered_df

,big_tile,number_code,direction_code,path_pch,path_bhf,path_ogs
0,hp,hp4500,HP40SE,NaN,/Users/ancazugo/Documents/PhD_Thesis/Tree_dete...,NaN
1,hp,hp5505,HP50NE,NaN,/Users/ancazugo/Documents/PhD_Thesis/Tree_dete...,NaN
2,hp,hp5005,HP50NW,NaN,/Users/ancazugo/Documents/PhD_Thesis/Tree_dete...,NaN
3,hp,hp5500,HP50SE,NaN,/Users/ancazugo/Documents/PhD_Thesis/Tree_dete...,NaN
4,hp,hp5000,HP50SW,NaN,/Users/ancazugo/Documents/PhD_Thesis/Tree_dete...,NaN
...,...,...,...,...,...,...
7297,tv,tv4595,TV49NE,/Users/ancazugo/Documents/PhD_Thesis/Tree_dete...,/Users/ancazugo/Documents/PhD_Thesis/Tree_dete...,/Users/ancazugo/Documents/PhD_Thesis/Tree_dete...
7298,tv,tv4095,TV49NW,NaN,/Users/ancazugo/Documents/PhD_Thesis/Tree_dete...,NaN
7299,tv,tv5595,TV59NE,/Users/ancazugo/Documents/PhD_Thesis/Tree_dete...,/Users/ancazugo/Documents/PhD_Thesis/Tree_dete...,/Users/ancazugo/Documents/PhD_Thesis/Tree_dete...
7300,tv,tv5095,TV59NW,/Users/ancazugo/Documents/PhD_Thesis/Tree_dete...,/Users/ancazugo/Documents/PhD_Thesis/Tree_dete...,/Users/ancazugo/Documents/PhD_Thesis/Tree_dete...


In [13]:
from shapely.ops import nearest_points

# Define a function that returns the closest point and distance for a given geometry
def find_closest_point_and_distance(reference_layer, compare_layer):

    single_polygon = compare_layer[compare_layer['height'] != 255].dissolve()
    geom = single_polygon.geometry.iloc[0]
    reference_point, closest_point = nearest_points(reference_layer, geom)
    distance = reference_layer.distance(closest_point)
    return pd.Series([distance], index=['distance_pch'])

In [24]:
def estimate_distance(row, crs=27700):
    
    print(row.number_code)
    bhf_tile_gdf = gpd.read_file(row.path_bhf).to_crs(crs)

    bhf_distance_gdf = gpd.GeoDataFrame()

    if row['path_ogs'] is not np.nan:    

        ogs_tile_gdf = gpd.read_file(row.path_ogs).to_crs(crs)
        nearest_ogs = gpd.sjoin_nearest(bhf_tile_gdf, ogs_tile_gdf, distance_col='distance_ogs', how='left')
        nearest_ogs = nearest_ogs[['id', 'function', 'distName1', 'distName2', 'distName3', 'distName4', 'distance_ogs', 'geometry']]

        bhf_distance_gdf = nearest_ogs

    if len(bhf_distance_gdf) == 0:
        return None
    
    bhf_distance_subtile_dir = BHF_OUT_DIR / 'bhf_ogs_distance' / row.big_tile
    bhf_distance_subtile_dir.mkdir(parents=True, exist_ok=True)
    bhf_distance_path = bhf_distance_subtile_dir / f"bhf_distance_{row.number_code}.geojson"
    bhf_distance_gdf.to_file(bhf_distance_path)

    return bhf_distance_gdf

In [14]:
def estimate_distance(row, crs=27700):
    
    print(row.number_code)
    bhf_tile_gdf = gpd.read_file(row.path_bhf).to_crs(crs)

    bhf_distance_gdf = gpd.GeoDataFrame()

    if row['path_pch'] is not np.nan:

        pch_tile_gdf = gpd.read_file(row.path_pch).to_crs(crs)
        closest_pch_gdf = bhf_tile_gdf.copy()
        # Calculate by point
        # closest_pch_gdf['distance_pch'] = closest_pch_gdf.apply(lambda row: find_closest_point_and_distance(row.geometry, pch_tile_gdf), axis=1)
        # Nearest index
        pch_tile_gdf = pch_tile_gdf[pch_tile_gdf['height'] != 255].dissolve()
        closest_pch_gdf = gpd.sjoin_nearest(bhf_tile_gdf, pch_tile_gdf, distance_col='distance_pch', how='left')
        bhf_distance_gdf = closest_pch_gdf[['distance_pch', 'geometry']]

    if row['path_ogs'] is not np.nan:    

        ogs_tile_gdf = gpd.read_file(row.path_ogs).to_crs(crs)
        nearest_ogs = gpd.sjoin_nearest(bhf_tile_gdf, ogs_tile_gdf, distance_col='distance_ogs', how='left')
        nearest_ogs = nearest_ogs[['id', 'function', 'distName1', 'distName2', 'distName3', 'distName4', 'distance_ogs', 'geometry']]

        if row['path_pch'] is not np.nan:
            bhf_distance_gdf = bhf_distance_gdf.merge(nearest_ogs, on='geometry', how='outer')
        else:
            bhf_distance_gdf = nearest_ogs

    if len(bhf_distance_gdf) == 0:
        return None

    bhf_distance_subtile_dir = bhf_distance_tile_dir / row.big_tile
    bhf_distance_subtile_dir.mkdir(parents=True, exist_ok=True)
    bhf_distance_path = bhf_distance_subtile_dir / f"bhf_distance_{row.number_code}.geojson"
    bhf_distance_gdf.to_file(bhf_distance_path)

    return bhf_distance_gdf

Tiles were processed normally until row #1286. Starting from row #1287 they will be fast-processed with nearest index for PCH distance

In [9]:
from concurrent.futures import ThreadPoolExecutor
cpu_cores_os = os.cpu_count()
cpu_cores_os

16

In [23]:
x = tile_paths_filtered_df[~tile_paths_filtered_df['path_ogs'].isna()]

In [ ]:
for index, row in tqdm(x.iterrows()):
    estimate_distance(row)
    # tl0020 tiene problemas con archivo pch.geojson (#6402) fiona.errors.DriverError: Failed to read GeoJSON data

In [195]:
%%time
with ThreadPoolExecutor(max_workers=cpu_cores_os) as executor:
    results = list(executor.map(estimate_distance, [row for _, row in tile_paths_filtered_df.iterrows()]))
    # results = list(tqdm(executor.map(estimate_distance, [row for _, row in tile_paths_filtered_df.iterrows()])))

ns4030
ns4040
ns4555
ns4565
ns4065
ns4560


## 3-rule (Three trees in sight)

## 30-rule (30% canopy cover in neighbourhood)

In [4]:
bhf_distance_tile_paths = sorted(list(bhf_distance_tile_dir.rglob('*.geojson')))

vrt_content = """<OGRVRTDataSource>
<OGRVRTUnionLayer name="union_layer">
{}
</OGRVRTUnionLayer>
</OGRVRTDataSource>"""

layer_template = """
    <OGRVRTLayer name="{layer_name}">
        <SrcDataSource>{file_path}</SrcDataSource>
    </OGRVRTLayer>"""

# Collect layer entries
layer_entries = []
for filename in bhf_distance_tile_paths:
    
    layer_name = filename.stem
    layer_entries.append(layer_template.format(layer_name=layer_name, file_path=filename))

# Format the complete VRT content
vrt_full_content = vrt_content.format(''.join(layer_entries))

bhf_distance_vrt_path = bhf_distance_tile_dir / 'bhf_distance.vrt'
# Write the VRT content to a file
with open(bhf_distance_vrt_path, 'w') as vrt_file:
    vrt_file.write(vrt_full_content)


In [5]:
from osgeo import ogr
data_source = ogr.Open(exe)
layer = data_source.GetLayer(0)

/Users/ancazugo/Documents/PhD_Thesis/Tree_detection/.venv/lib/python3.12/site-packages/osgeo/ogr.py:593: FutureWarning: Neither ogr.UseExceptions() nor ogr.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


In [34]:
bhf_distance_ogs_dir = BHF_OUT_DIR / 'bhf_ogs_distance'
bhf_distance_ogs_dir = sorted(list(bhf_distance_ogs_dir.rglob('*.geojson')))

In [36]:
# Perform a spatial join
bhf_imd_ogs_sjoin_lst = []
for path in tqdm(bhf_distance_ogs_dir):
    bhf_distance_ex = gpd.read_file(path)
    buildings_with_neighborhoods = gpd.sjoin(bhf_distance_ex.copy().to_crs(4326), imd_gb_gpd[['LSOA', 'geometry']], how="inner")
    bhf_imd_ogs_sjoin_lst.append(buildings_with_neighborhoods)

100%|██████████| 2879/2879 [20:52<00:00,  2.30it/s]


In [38]:
bhf_ogs_imd_agg = pd.concat(bhf_imd_ogs_sjoin_lst).groupby('LSOA').agg({
    'distance_ogs': ['mean', 'count', 'sum'],
    # Add more metrics as needed
}).reset_index()
bhf_ogs_imd_agg.columns = [' '.join(col).strip() for col in bhf_ogs_imd_agg.columns.values]
bhf_ogs_imd_agg.drop(columns=['distance_ogs sum'], inplace=True)

In [41]:
bhf_imd_gb_agg_gpd_path = IMD_OUT_DIR / 'imd_distance_aggregated.geojson'
# bhf_imd_gb_agg_gpd =gpd.read_file(bhf_imd_gb_agg_gpd_path)
bhf_imd_gb_agg_gpd.drop(columns=['distance_ogs mean', 'distance_ogs count'])

,LSOA,LANAME,Rank,SOA_pct,SOA_decile,LA_Rank,LA_pct,LA_decile,distance_pch mean,distance_pch count,geometry
0,E01000001,City of London,29199,89,9,208,66,7,4.193399,33,"POLYGON ((-0.09729 51.52158, -0.09652 51.52027..."
1,E01000002,City of London,30379,92,10,208,66,7,17.562168,61,"POLYGON ((-0.08813 51.51941, -0.08929 51.51752..."
2,E01000003,City of London,14915,45,5,208,66,7,7.158265,24,"POLYGON ((-0.09679 51.52325, -0.09647 51.52282..."
3,E01000005,City of London,8678,26,3,208,66,7,6.747280,65,"POLYGON ((-0.07323 51.51000, -0.07553 51.50974..."
4,E01000006,Barking and Dagenham,14486,44,5,5,2,1,1.853085,186,"POLYGON ((0.09115 51.53909, 0.09326 51.53787, ..."
...,...,...,...,...,...,...,...,...,...,...,...
38309,S01013477,West Lothian,2464,35,4,16,50,5,4.331948,140,"POLYGON ((-3.46323 55.93434, -3.46319 55.93408..."
38310,S01013478,West Lothian,3681,53,6,16,50,5,4.415355,223,"POLYGON ((-3.48355 55.93733, -3.48354 55.93728..."
38311,S01013479,West Lothian,1423,20,3,16,50,5,2.727565,214,"POLYGON ((-3.46664 55.93627, -3.46652 55.93622..."
38312,S01013480,West Lothian,3291,47,5,16,50,5,6.597745,99,"POLYGON ((-3.46259 55.93774, -3.46243 55.93725..."


In [47]:
bhf_imd_gb_agg_gpd_path

PosixPath('/Users/ancazugo/Documents/PhD_Thesis/Tree_detection/data/output/vector/imd/imd_distance_aggregated.geojson')

In [43]:
bhf_imd_distance_correct_gdf = bhf_imd_gb_agg_gpd.merge(bhf_ogs_imd_agg, how='left')

In [45]:
bhf_imd_distance_correct_gdf.to_file(bhf_imd_gb_agg_gpd_path)

In [31]:
bhf_distance_tile_paths = sorted(list(bhf_distance_tile_dir.rglob('*.geojson')))
imd_gb_gpd = gpd.read_file(imd_gb_gdf_path)


In [26]:
# Perform a spatial join
bhf_imd_sjoin_lst = []
for path in tqdm(bhf_distance_tile_paths):
    bhf_distance_ex = gpd.read_file(path)
    buildings_with_neighborhoods = gpd.sjoin(bhf_distance_ex.copy().to_crs(4326), imd_gb_gpd[['LSOA', 'geometry']], how="inner")
    bhf_imd_sjoin_lst.append(buildings_with_neighborhoods)

100%|██████████| 6138/6138 [48:03<00:00,  2.13it/s]    


In [52]:
bhf_imd_agg = pd.concat(bhf_imd_sjoin_lst).groupby('LSOA').agg({
    'distance_pch': ['mean', 'count', 'sum'],
    'distance_ogs': ['mean', 'count', 'sum'],
    # Add more metrics as needed
}).reset_index()
bhf_imd_agg.columns = [' '.join(col).strip() for col in bhf_imd_agg.columns.values]

bhf_imd_gb_agg_gpd =  imd_gb_gpd.merge(bhf_imd_agg, how='inner', on='LSOA')\
    .drop(columns=['StdAreaHa', 'StdAreaKm2', 'Shape__Are', 'Shape__Len', 'Shape_Leng',
                   'Shape_Area', 'TotPop2011', 'ResPop2011', 'HHCnt2011', 'IMDRank',
                   'IMDDecil', 'lsoa11nm', 'Name', 'FID', 'distance_pch sum', 'distance_ogs sum'])


bhf_imd_gb_agg_gpd_path = IMD_OUT_DIR / 'imd_distance_aggregated.geojson'
bhf_imd_gb_agg_gpd.to_file(bhf_imd_gb_agg_gpd_path) 

In [51]:
bhf_imd_gb_agg_gpd.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [ ]:
import leafmap
import rasterio
import matplotlib.pyplot as plt
bhf_imd_gb_agg_gpd_path = IMD_OUT_DIR / 'imd_distance_aggregated.geojson'
# Example of getting hex values of colors from a ColorBrewer colormap
# cmap_greens = plt.get_cmap("RdYlBu", 10)
# greens_colors_hex = [plt.cm.colors.rgb2hex(cmap_greens(i / 9)) for i in range(10)]  # Adjust 10 to get more or fewer colors
# cmap_rdylbu = plt.get_cmap("RdYlBu", 10)
# rdylbu_colors_hex = [plt.cm.colors.rgb2hex(cmap_rdylbu(i / 9)) for i in range(10)]  # Adjust 10 to get more or fewer colors

m = leafmap.Map(center=[51.49562152904102, -0.14623995675458693], zoom=14)
m.add_geojson(str(bhf_imd_gb_agg_gpd_path), layer_name='SOA_decile')
m.add_geojson(str(bhf_imd_gb_agg_gpd_path), layer_name='distance_ogs mean')
# m.add_geojson(str(bhf_imd_gb_agg_gpd_path), layer_name='SOA_decile', fill_colors=rdylbu_colors_hex)
# m.add_geojson(str(bhf_imd_gb_agg_gpd_path), layer_name='distance_ogs mean', fill_colors=greens_colors_hex)
# m.split_map(
#     left_label="Deprivation",
#     right_label="Proximity to park"
# )
m.split_map()
m

## 300-rule (300 m from the closest public park)

In [18]:
import rasterio

vrt_masked = rasterio.open(pch_binary_vrt_path)
bhf_imd_gb_agg_gpd = gpd.read_file(IMD_OUT_DIR / 'imd_distance_aggregated.geojson')
vrt_masked

<open DatasetReader name='/Users/ancazugo/Documents/PhD_Thesis/Tree_detection/data/output/raster/pch/pch_binary/ps_PSScene4Band_2019_binary.vrt' mode='r'>

In [25]:
import geopandas as gpd
import rasterio
from rasterio.features import geometry_mask
from rasterstats import zonal_stats

def calculate_overlap_area(raster_path, gdf):
    # Read the raster and vector data
    with rasterio.open(raster_path) as src:
        affine = src.transform
        array = src.read(1)  # Read the first band where binary values are stored

    # Ensure vector and raster have the same CRS
    if gdf.crs != src.crs:
        gdf = gdf.to_crs(src.crs)

    # Calculate zonal statistics
    stats = zonal_stats(gdf, array, affine=affine, stats="count", nodata=0)

    # Calculate the area using the count of pixels and the area of a single pixel
    pixel_area = src.res[0] * src.res[1]  # resolution in x multiplied by resolution in y
    gdf['overlap_area'] = [s['count'] * pixel_area for s in stats]

    return gdf


In [26]:
overlap_gdf = calculate_overlap_area(pch_binary_vrt_path, bhf_imd_gb_agg_gpd)

In [38]:
overlap_gdf['canopy_cover'] = 100 * overlap_gdf.overlap_area / overlap_gdf.area

/var/folders/gr/150_8wbn02s9rdqqvb4pbblh0000gn/T/ipykernel_8410/3600944182.py:1: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  overlap_gdf['canopy_cover'] = 100 * overlap_gdf.overlap_area / overlap_gdf.area


In [42]:
overlap_gdf.to_file(IMD_OUT_DIR / 'imd_distance_aggregated_canopy_cover.geojson')

# Notes
## Data Sources

- PCH: [Liu et al. 2023](https://zenodo.org/records/8154445) (UK - 2019)
- BUA: [Digimap](https://digimap.edina.ac.uk/) (GB - December 2022)
- GBG: [Digimap](https://digimap.edina.ac.uk/) (GB - 2012)
- BHF: [Digimap](https://digimap.edina.ac.uk/) (GB - October 2017)
- OGS: [Digimap](https://digimap.edina.ac.uk/) (GB - October 2017)
- IMD: [Consumer Data Research Center](https://data.cdrc.ac.uk/dataset/index-multiple-deprivation-imd) (UK - 2019-2020)
  - LSOA Boundaries (England and Wales) [Ministry of Housing, Communities and Local Government](https://data-communities.opendata.arcgis.com/datasets/6bced6c6f81448cf9692ed3f472b11ce/explore?location=52.661142%2C-2.243978%2C7.00) (2019)
  - Zone Boundaries (Scotland) [www.data.gov.uk](https://www.data.gov.uk/dataset/ab9f1f20-3b7f-4efa-9bd2-239acf63b540/data-zone-boundaries-2011) (2011)

## CRS
All the data processing steps were performed using a standard CRS of *EPSG:4326* (degrees) for all layers. To calculate distances, vector layers were transformed to *EPSG:27700* (metres).

In [155]:
print('PCH:', pch_crs)
print('BUA:', bua_crs)
print('GBG:', gbg_crs)
print('BHF:', bhf_crs)
print('OGS:', ogs_crs)

PCH: EPSG:4326
BUA: EPSG:27700
GBG: PROJCS["OSGB36 / British National Grid",GEOGCS["OSGB36",DATUM["Ordnance_Survey_of_Great_Britain_1936",SPHEROID["Airy 1830",6377563.396,299.3249646,AUTHORITY["EPSG","7001"]],AUTHORITY["EPSG","6277"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",49],PARAMETER["central_meridian",-2],PARAMETER["scale_factor",0.999601272],PARAMETER["false_easting",400000],PARAMETER["false_northing",-100000],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
BHF: EPSG:27700
OGS: EPSG:27700
